In [1]:
!pip install polars

import polars as pl

In [2]:
import os
import random
import sys
import cv2
import tqdm
import json
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import multilabel_confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [3]:

!nvidia-smi

Fri Mar 13 15:04:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               On  |   00000000:4F:00.0 Off |                  Off |
| 31%   62C    P2            220W /  230W |   15381MiB /  24564MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os
os.chdir('/workspace')

In [5]:
import pandas as pd
import os

BASE_PATH = 'vindr_mammogram'
meta_path = os.path.join(BASE_PATH, 'metadata.csv')

device = pd.read_csv(meta_path)
device.head()

,SOP Instance UID,Series Instance UID,SOP Instance UID.1,Patient's Age,View Position,Image Laterality,Photometric Interpretation,Rows,Columns,Imager Pixel Spacing,...,Pixel Padding Value,Pixel Padding Range Limit,Window Center,Window Width,Rescale Intercept,Rescale Slope,Rescale Type,Window Center & Width Explanation,Manufacturer,Manufacturer's Model Name
0,d8125545210c08e1b1793a5af6458ee2,b36517b9cbbcfd286a7ae04f643af97a,d8125545210c08e1b1793a5af6458ee2,053Y,CC,L,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1662,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
1,290c658f4e75a3f83ec78a847414297c,b36517b9cbbcfd286a7ae04f643af97a,290c658f4e75a3f83ec78a847414297c,053Y,MLO,L,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1664,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
2,cd0fc7bc53ac632a11643ac4cc91002a,b36517b9cbbcfd286a7ae04f643af97a,cd0fc7bc53ac632a11643ac4cc91002a,053Y,CC,R,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1600,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
3,71638b1e853799f227492bfb08a01491,b36517b9cbbcfd286a7ae04f643af97a,71638b1e853799f227492bfb08a01491,053Y,MLO,R,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1654,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
4,dd9ce3288c0773e006a294188aadba8e,d931832a0815df082c085b6e09d20aac,dd9ce3288c0773e006a294188aadba8e,042Y,CC,L,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1580,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration


In [6]:
BASE_PATH = 'vindr_mammogram'
IMG_DIR = os.path.join(BASE_PATH, 'mammo_processed_merged1') 
csv_files = [f for f in os.listdir(IMG_DIR) if f.endswith('.csv')]

# Print results
if csv_files:
    print(f"Found {len(csv_files)} CSV file(s) in {IMG_DIR}:")
    for csv_file in csv_files:
        print(f"{csv_file}")
else:
    print(f"No CSV files found in {IMG_DIR}")


Found 1 CSV file(s) in vindr_mammogram/mammo_processed_merged1:
mammo_processed_merged.csv


In [ ]:
df = pl.read_csv("vindr_mammogram/mammo_processed_merged1/mammo_processed_merged.csv")
df = df.to_pandas()
df["merged_image_path"] = (
    df["merged_image_path"]
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
)
density_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3}

df['density'] = df['cc_breast_density'].str[-1].map(density_mapping)
df.head()



In [ ]:
import pandas as pd

device_info = device[['SOP Instance UID', "Manufacturer's Model Name"]].copy()
device_info.columns = ['image_id', 'device_model']

df = df.merge(device_info, left_on='cc_image_id', right_on='image_id', how='left')
df = df.drop('image_id', axis=1)

device_mapping = {
    'Mammomat Inspiration': 0,
    'Planmed Nuance': 1,
    'GIOTTO CLASS': 2,
    'GIOTTO IMAGE 3DL': 2
}

df['device_id'] = df['device_model'].map(device_mapping)

print(df['device_model'].value_counts())
print(df['device_id'].value_counts())
print(df.shape)

In [ ]:
def get_combined_finding_6class(cc_findings, mlo_findings, cc_birads, mlo_birads):
    if isinstance(cc_findings, str):
        cc_findings = eval(cc_findings) if cc_findings else []
    if isinstance(mlo_findings, str):
        mlo_findings = eval(mlo_findings) if mlo_findings else []
    
    cc_findings = cc_findings or []
    mlo_findings = mlo_findings or []
    all_findings = set(cc_findings + mlo_findings)
    
    if len(all_findings) > 1 and 'No Finding' in all_findings:
        all_findings.remove('No Finding')
    
    high_suspicion_structural = {
        'Architectural Distortion',
        'Skin Thickening',
        'Skin Retraction',
        'Nipple Retraction'
    }
    
    asymmetry_findings = {
        'Focal Asymmetry',
        'Global Asymmetry',
        'Asymmetry'
    }
    
    has_mass = 'Mass' in all_findings
    has_calc = 'Suspicious Calcification' in all_findings
    has_structural = bool(all_findings & high_suspicion_structural)
    has_asymmetry = bool(all_findings & asymmetry_findings)
    has_lymph = 'Suspicious Lymph Node' in all_findings
    
    def parse_birads(birads_str):
        if pd.isna(birads_str) or birads_str == '':
            return 0
        if isinstance(birads_str, str):
            try:
                return int(birads_str.strip().split()[-1])
            except:
                return 0
        return int(birads_str)
    
    cc_birads_num = parse_birads(cc_birads)
    mlo_birads_num = parse_birads(mlo_birads)
    max_birads = max(cc_birads_num, mlo_birads_num)
    
    if not all_findings or all_findings == {'No Finding'}:
        if max_birads == 1:
            return 0
        elif max_birads == 2:
            return 1
        else:
            return 1 if max_birads == 3 else 4
    
    if (has_mass and has_calc) or has_lymph:
        return 4  # Complex/Combined
    
    # Priority 5: Architectural distortions (without mass)
    if has_structural:
        return 5
    
    # Priority 3: Mass (isolated)
    if has_mass:
        return 3
    
    # Priority 2: Calcification (isolated)
    if has_calc:
        return 2
    
    if has_asymmetry and len(all_findings) == 1:
        return -1
    
    if has_asymmetry and len(all_findings) > 1:
        return 5
    
    print(f"Warning: Unknown finding combination: {all_findings}, BIRADS: {max_birads}")
    return 5

df['finding'] = df.apply(
    lambda row: get_combined_finding_6class(
        row['cc_finding_categories'], 
        row['mlo_finding_categories'],
        row['cc_breast_birads'],
        row['mlo_breast_birads']
    ),
    axis=1
)
df.drop(df[df['finding'] == -1].index, inplace=True)
df['finding'].value_counts().sort_index()

In [ ]:
inbreast_df = pd.read_csv("inbreast_data/INbreast_merged/merged_metadata.csv")
inbreast_df["merged_image_path"] = (
    inbreast_df["merged_image_path"]
    .str.replace("INbreast Release 1.0", "inbreast_data", regex=False)
    .str.replace("vindr_original_data", "inbreast_data", regex=False))
inbreast_df['birads'] = inbreast_df['birads'].replace({'4a': '4', '4b': '4', '4c': '4','6':'5'})
inbreast_df['birads'] = (inbreast_df['birads'].astype(int) - 1).astype(int)
def birads_to_binary(birads):
    if birads ==0 :
        return 0 
    else:
        return 1 
inbreast_df['label'] = inbreast_df['birads'].apply(birads_to_binary)
inbreast_df['density'] = 0
inbreast_df['device_id'] = 0
inbreast_df['finding'] = 0
inbreast_df.head()

In [ ]:
def birads_to_label(birads_category):
    """Convert BI-RADS categories to numerical labels 0-4 (for 5 classes)"""
    birads_num = int(birads_category.replace(" ", "")[-1])
    return birads_num - 1
df['birads'] = df['cc_breast_birads'].apply(birads_to_label)

In [ ]:
def birads_to_binary(birads):
    return 0 if birads in ['BI-RADS 1'] else 1 
df['label'] = df['cc_breast_birads'].apply(birads_to_binary)

In [ ]:
df['cc_breast_birads'].value_counts()

In [ ]:
df['cc_breast_density'].value_counts()

In [ ]:
data = df[df['split'] == 'training']
test_df = df[df['split'] == 'test']

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

views_per_study    = data.groupby('study_id').size()
complete_studies   = views_per_study[views_per_study == 2].index
incomplete_studies = views_per_study[views_per_study != 2].index

study_meta = (
    data[data['study_id'].isin(complete_studies)]
    .groupby('study_id')
    .agg({
        'cc_breast_birads': 'first',
        'finding':          'first',
        'device_id':        'first'
    })
    .reset_index()
)

study_meta['stratify_key'] = (
    study_meta['cc_breast_birads'].astype(str) + '_' +
    study_meta['finding'].astype(str)           + '_' +
    study_meta['device_id'].astype(str)
)

key_counts = study_meta['stratify_key'].value_counts()
rare_keys  = key_counts[key_counts < 2].index
study_meta['stratify_key'] = study_meta['stratify_key'].apply(
    lambda k: 'rare' if k in rare_keys else k
)

train_studies, val_studies = train_test_split(
    study_meta['study_id'],
    test_size    = 0.1,
    stratify     = study_meta['stratify_key'],
    random_state = 423
)

train_studies = pd.concat([
    pd.Series(train_studies),
    pd.Series(incomplete_studies)
]).reset_index(drop=True)

train_df = data[data['study_id'].isin(train_studies)].copy()
val_df   = data[data['study_id'].isin(val_studies)].copy()

# fix incomplete in train — keep as single view (don't drop)
train_views = train_df.groupby('study_id').size()
if (train_views != 2).any():
    single_view = train_views[train_views != 2].index
    train_df    = train_df[~train_df['study_id'].isin(single_view)].copy()
    single_df   = data[data['study_id'].isin(single_view)].copy()
    train_df    = pd.concat([train_df, single_df], ignore_index=True)

print(f"Train studies: {train_df['study_id'].nunique()} | Train samples: {len(train_df)}")
print(f"Val   studies: {val_df['study_id'].nunique()}   | Val   samples: {len(val_df)}")

In [ ]:
train_df['density'].value_counts()

In [ ]:
val_df['density'].value_counts()

In [ ]:
train_df.shape

In [ ]:
import numpy as np
import cv2
from PIL import Image
import torchvision.transforms as transforms
import random
import torch


def get_transforms(img_size=(512, 512)):
    """Enhanced mammogram preprocessing with medical imaging considerations"""
    
    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        
        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=10,
                translate=(0.05, 0.05),
                scale=(0.9, 1.05),
                shear=6
            )
        ], p=0.6),
        
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([
            transforms.RandomPerspective(distortion_scale=0.1, p=1.0)
        ], p=0.1),
        
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=5.0, sigma=3.0)
        ], p=0.2),
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.95, 1.05),
                contrast=(0.9, 1.1)
            )
        ], p=0.6),
        transforms.Lambda(lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2)) 
                         if random.random() < 0.4 else x),
        

        
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
        ], p=0.2),
        
                # NOISE AUGMENTATIONS
        transforms.Lambda(lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.03)) 
                         if random.random() < 0.4 else x),
        
        transforms.Lambda(lambda x: add_speckle_noise(x, std=random.uniform(0.01, 0.03)) 
                         if random.random() < 0.2 else x),
        transforms.ToTensor(),
        
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    return train_transform, val_transform

def add_gaussian_noise(image, mean=0, std=0.02):
    """Gaussian noise - electronic noise in imaging sensors"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(mean, std, img_array.shape)
        noisy_img = np.clip(img_array + noise, 0, 1)
        return Image.fromarray((noisy_img * 255).astype(np.uint8))
    return image


def add_speckle_noise(image, std=0.03):
    """Speckle noise - multiplicative noise common in mammography"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(0, std, img_array.shape)
        noisy_img = img_array + img_array * noise
        return Image.fromarray((np.clip(noisy_img, 0, 1) * 255).astype(np.uint8))
    return image

def adjust_gamma(image, gamma=1.0):
    """
    Gamma correction - handles tissue density variation
    Gamma < 1 = brighter, > 1 = darker
    """
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        gamma_corrected = np.power(img_array, gamma)
        return Image.fromarray((gamma_corrected * 255).astype(np.uint8))
    return image

In [ ]:
import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import pandas as pd
from PIL import Image
import os
from torchvision import transforms
import matplotlib.pyplot as plt
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
import random

class VinDrMammoDataset(Dataset):
    def __init__(self, dataframe, train_light, val_transform, mode="train"):
        self.df = dataframe.reset_index(drop=True)
        self.train_light = train_light
        self.val_transform = val_transform
        self.mode = mode

        counts = self.df["label"].value_counts().to_dict()
        print(f"{mode.upper()} class distribution:", counts)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["merged_image_path"]).convert("RGB")
        # image=preprocess_mammogram(image)
        label = int(row["label"])

        birads=row['birads']
        if self.mode == "train":
            image_tensor = self.train_light(image)
        else:
            image_tensor = self.val_transform(image)
            
        density = row['density'] if 'density' in row else 0 
        density_tensor = torch.tensor(density, dtype=torch.long)
        device = row['device_id'] if 'device_id' in row else 0 
        device_tensor = torch.tensor(device, dtype=torch.long)
        finding = row['finding'] if 'finding' in row else 0 
        finding = torch.tensor(finding, dtype=torch.long)
        return image_tensor, torch.tensor(label, dtype=torch.long)
        # return row["merged_image_path"], image_tensor, torch.tensor(label, dtype=torch.long),density_tensor,device_tensor,finding

def create_data_loaders(train_df, val, test_df, inbreast_df, batch_size=8, img_size=(512, 512)):
    """Create train and val dataloaders with weighted sampling"""
    train_light, val_transform = get_transforms(img_size)

    train_dataset = VinDrMammoDataset(
        train_df, train_light, val_transform, mode="train"
    )
    val_dataset = VinDrMammoDataset(
        val_df, train_light, val_transform, mode="val"
    )
    test_dataset = VinDrMammoDataset(
        test_df, train_light, val_transform, mode="val"
    )
    inbreast_dataset = VinDrMammoDataset(
        inbreast_df, train_light,val_transform, mode="val"
    )
    # labels = train_df['label'].values 
    # sampler = MajorityUnderSampler(labels, target_ratio=1.0).get_sampler()  # 1:1 balance
    
    labels = train_df['label'].values
    unique_classes, class_counts = np.unique(labels, return_counts=True)
    
    β = 0.5 # 0.5–0.8 typical; 1.0 means full inverse, 0.0 means no balancing
    class_weights = (1.0 / class_counts) ** β
    class_weights = class_weights / class_weights.sum() * len(unique_classes)
    
    sample_weights = class_weights[labels]
    print("Class counts:", dict(zip(unique_classes, class_counts)))
    print("Smoothed class weights:", np.round(class_weights, 3))
    
    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(sample_weights).float(),
        num_samples=len(sample_weights),
        replacement=True
    )

    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=sampler,
        # shuffle=True,
        num_workers=6,
        pin_memory=True,
        drop_last=True  
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=6,
        pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=6,
        pin_memory=True
    )

    inbreast_loader = DataLoader(
        inbreast_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=6,
        pin_memory=True
    )
    return train_loader, val_loader,test_loader,inbreast_loader

batch_size=16
train_loader,val_loader,test_loader,inbreast_loader = create_data_loaders(
    train_df,val_df, test_df, inbreast_df,
    batch_size=batch_size,
    img_size=(512, 512)
)

In [ ]:
def visualize_batch(batch, n_cols=4, apply_inverse_normalize=True):
    """Visualize a batch of images with their labels"""
    # _,images, labels,_,_,_ = batch
    images, labels= batch
    batch_size = images.shape[0]
    n_rows = int(np.ceil(batch_size / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3*n_rows))
    axes = axes.ravel() if isinstance(axes, np.ndarray) else [axes]
    
    for i in range(batch_size):
        img = images[i]
        label = labels[i].item()
        
        # Inverse normalization if needed
        if apply_inverse_normalize:
            img = img.permute(1, 2, 0).numpy()
            # img=img*255
            img = (img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]) * 255
            img = img.mean(axis=2).astype(np.uint8)
        else:
            img = img.permute(1, 2, 0).numpy().astype(np.uint8)
        
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(f"Label: {label}")
        axes[i].axis('off')
    
    # Hide empty subplots
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize samples
print("Visualizing training samples:")
# Get a batch
train_batch = next(iter(train_loader))
visualize_batch(train_batch)

print("\nVisualizing validation samples:")
val_batch = next(iter(val_loader))
visualize_batch(val_batch)

In [ ]:

from tqdm import tqdm
import torch.nn as nn
from torchvision import models
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torch.optim import Adam
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
"""
Multi-Scale FPN Backbone for Mammography Binary Classification
==============================================================

Verified stage dimensions (input 512x512):

EfficientNet-B3:
  features[:4]  -> ch=48,   spatial=64x64    (C3)
  features[:5]  -> ch=96,   spatial=32x32    (C4)   ← was wrong before
  features[:9]  -> ch=1536, spatial=16x16    (C5)

ConvNeXt-Base:
  features[:2]  -> ch=128,  spatial=128x128  (C3)
  features[:4]  -> ch=256,  spatial=64x64    (C4)
  features[:8]  -> ch=1024, spatial=16x16    (C5)

FPN merges C3/C4/C5 → P3/P4/P5 (each 256ch)
Pool each → flatten → cat → [256*3=768] → classifier head

Directory layout per backbone:
    base_dir/backbone_name/
        best_model.pth
        training_log.txt
        summary.txt
        vindr_val/      VinDr-Val-Epoch{N}_report.txt  (on every val F1 improvement)
        vindr_test/     VinDr-Test_report.txt
        inbreast_test/  INbreast_report.txt
"""

import os
import gc
import time
import random
from typing import Tuple, Dict, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.amp import GradScaler
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
import torchvision.models as tvm


# ─────────────────────────────────────────────────────────────────────────────
# SEED
# ─────────────────────────────────────────────────────────────────────────────

def set_seed(seed: int = 2024):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ─────────────────────────────────────────────────────────────────────────────
# LOSS
# ─────────────────────────────────────────────────────────────────────────────

class AsymmetricFocalLoss(nn.Module):
    """
    Asymmetric focal loss.
      gamma_pos = 1  : mild down-weighting of easy cancer samples
      gamma_neg = 2  : stronger down-weighting of easy benign samples
      alpha     = 0.8: up-weights cancer class
    """
    def __init__(
        self,
        gamma_neg      : float = 2.0,
        gamma_pos      : float = 1.0,
        alpha          : float = 0.8,
        label_smoothing: float = 0.0,
    ):
        super().__init__()
        self.gamma_neg       = gamma_neg
        self.gamma_pos       = gamma_pos
        self.alpha           = alpha
        self.label_smoothing = label_smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.long()
        ce = F.cross_entropy(
            logits, targets, reduction="none",
            label_smoothing=self.label_smoothing,
        )
        pt    = torch.exp(-ce)
        gamma = torch.where(targets == 1,
                            torch.full_like(ce, self.gamma_pos),
                            torch.full_like(ce, self.gamma_neg))
        alpha = torch.where(targets == 1,
                            torch.full_like(ce, self.alpha),
                            torch.full_like(ce, 1.0 - self.alpha))
        loss = (alpha * (1.0 - pt).pow(gamma) * ce).mean()
        if torch.isnan(loss) or torch.isinf(loss):
            return logits.sum() * 0.0
        return loss


# ─────────────────────────────────────────────────────────────────────────────
# REPORT SAVING
# ─────────────────────────────────────────────────────────────────────────────

def save_report(
    save_dir     : str,
    folder_name  : str,
    dataset_name : str,
    labels,
    preds,
    class_names  : tuple,
    extra_metrics: dict = None,
) -> str:
    """
    Write classification report + confusion matrix to a .txt file.

    Path: save_dir/folder_name/{dataset_name}_report.txt
    """
    out_dir  = os.path.join(save_dir, folder_name)
    os.makedirs(out_dir, exist_ok=True)
    txt_path = os.path.join(out_dir, f"{dataset_name}_report.txt")

    cm  = confusion_matrix(labels, preds)
    rep = classification_report(
        labels, preds, target_names=list(class_names), zero_division=0
    )
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="binary", pos_label=1, zero_division=0)

    tn = fp = fn = tp = 0
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    sep = "=" * 62
    with open(txt_path, "w") as f:
        f.write(f"{sep}\n")
        f.write(f"  RESULTS: {dataset_name}\n")
        f.write(f"{sep}\n\n")
        f.write(f"Accuracy    : {acc:.4f}\n")
        f.write(f"F1 (cancer) : {f1:.4f}\n")
        f.write(f"Sensitivity : {sens:.4f}\n")
        f.write(f"Specificity : {spec:.4f}\n")
        if extra_metrics:
            for k, v in extra_metrics.items():
                f.write(f"{k:<14}: {v}\n")
        f.write(f"\n{sep}\n")
        f.write("Confusion Matrix\n")
        f.write(f"{sep}\n")
        col_w = 16
        header = f"{'':>18}" + "".join(
            f"Pred {c:>{col_w - 5}}" for c in class_names
        )
        f.write(header + "\n")
        for i, row in enumerate(cm):
            row_str = "".join(f"{v:>{col_w}}" for v in row)
            f.write(f"True {class_names[i]:>12}  {row_str}\n")
        f.write(f"\n{sep}\n")
        f.write("Classification Report\n")
        f.write(f"{sep}\n")
        f.write(rep)
        f.write(f"\n{sep}\n")

    print(f"  [saved] {txt_path}")
    return txt_path


# ─────────────────────────────────────────────────────────────────────────────
# PRINT HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def print_epoch_summary(
    split      : str,
    loss       : float,
    acc        : float,
    f1         : float,
    preds,
    labels,
    class_names: tuple = ("Benign", "Malignant"),
):
    sep = "-" * 62
    print(f"\n{sep}")
    print(f"  {split.upper()}  |  Loss={loss:.4f}  Acc={acc:.4f}  F1={f1:.4f}")
    print(sep)
    print(classification_report(labels, preds,
          target_names=list(class_names), zero_division=0))
    cm = confusion_matrix(labels, preds)
    print(f"Confusion Matrix:")
    header = f"{'':>14}" + "  ".join(f"Pred {c:>10}" for c in class_names)
    print(header)
    for i, row in enumerate(cm):
        print(f"True {class_names[i]:>8}  " + "  ".join(f"{v:>14}" for v in row))
    print(sep)


# ─────────────────────────────────────────────────────────────────────────────
# FPN
# ─────────────────────────────────────────────────────────────────────────────

class ConvBNAct(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int, s: int = 1, p: int = 0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SimpleFPN(nn.Module):
    """
    Top-down FPN.
    Inputs : C3 (large), C4 (medium), C5 (small/deepest)
    Outputs: P3, P4, P5  — all with out_channels channels

    Merge rule:
      P5 = lateral5(C5)
      P4 = lateral4(C4) + upsample(P5 → C4 size)
      P3 = lateral3(C3) + upsample(P4 → C3 size)
    """
    def __init__(self, in_channels_list: List[int], out_channels: int = 256):
        super().__init__()
        c3_ch, c4_ch, c5_ch = in_channels_list

        self.lat3 = nn.Conv2d(c3_ch, out_channels, kernel_size=1)
        self.lat4 = nn.Conv2d(c4_ch, out_channels, kernel_size=1)
        self.lat5 = nn.Conv2d(c5_ch, out_channels, kernel_size=1)

        self.out3 = ConvBNAct(out_channels, out_channels, k=3, p=1)
        self.out4 = ConvBNAct(out_channels, out_channels, k=3, p=1)
        self.out5 = ConvBNAct(out_channels, out_channels, k=3, p=1)

    def forward(
        self,
        c3: torch.Tensor,
        c4: torch.Tensor,
        c5: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        p5 = self.lat5(c5)
        p4 = self.lat4(c4) + F.interpolate(p5, size=c4.shape[-2:], mode="nearest")
        p3 = self.lat3(c3) + F.interpolate(p4, size=c3.shape[-2:], mode="nearest")
        return self.out3(p3), self.out4(p4), self.out5(p5)


# ─────────────────────────────────────────────────────────────────────────────
# MULTI-SCALE BACKBONE
# ─────────────────────────────────────────────────────────────────────────────

class MultiScaleBackbone(nn.Module):
    """
    Backbone → C3/C4/C5 → FPN → GAP each → concat → classifier

    Verified channel dims (512×512 input):

    EfficientNet-B3:
      stage1 = features[:4]   C3: 48ch  @ 64×64
      stage2 = features[4:5]  C4: 96ch  @ 32×32
      stage3 = features[5:]   C5: 1536ch @ 16×16

    ConvNeXt-Base:
      stage1 = features[:2]   C3: 128ch  @ 128×128
      stage2 = features[2:4]  C4: 256ch  @ 64×64
      stage3 = features[4:]   C5: 1024ch @ 16×16

    Classifier input dim = fpn_dim × 3  (P3 + P4 + P5 pooled and concatenated)
    """

    def __init__(
        self,
        backbone_name: str  = "efficientnet_b3",
        num_classes  : int  = 2,
        fpn_dim      : int  = 256,
        head_dim     : int  = 512,
        dropout      : float = 0.4,
        pretrained   : bool  = True,
    ):
        super().__init__()
        self.backbone_name = backbone_name

        # ── Stage splits (verified on 512×512 input) ─────────────────────────
        if backbone_name == "efficientnet_b3":
            net = tvm.efficientnet_b3(
                weights=tvm.EfficientNet_B3_Weights.DEFAULT if pretrained else None
            )
            # features[:4]  → 48ch  @ 64×64
            # features[4:5] → 96ch  @ 32×32
            # features[5:]  → 1536ch @ 16×16
            self.stage1 = net.features[:4]
            self.stage2 = net.features[4:5]
            self.stage3 = net.features[5:]
            c3_dim, c4_dim, c5_dim = 48, 96, 1536

        elif backbone_name == "convnext_base":
            net = tvm.convnext_base(
                weights=tvm.ConvNeXt_Base_Weights.DEFAULT if pretrained else None
            )
            # features[:2]  → 128ch  @ 128×128
            # features[2:4] → 256ch  @ 64×64
            # features[4:]  → 1024ch @ 16×16
            self.stage1 = nn.Sequential(*list(net.features)[:2])
            self.stage2 = nn.Sequential(*list(net.features)[2:4])
            self.stage3 = nn.Sequential(*list(net.features)[4:])
            c3_dim, c4_dim, c5_dim = 128, 256, 1024

        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}. "
                             f"Choose 'efficientnet_b3' or 'convnext_base'.")

        # ── FPN ───────────────────────────────────────────────────────────────
        self.fpn  = SimpleFPN(
            in_channels_list=[c3_dim, c4_dim, c5_dim],
            out_channels=fpn_dim,
        )
        self.pool = nn.AdaptiveAvgPool2d(1)   # GAP each P level → (B, fpn_dim, 1, 1)

        # ── Classifier ────────────────────────────────────────────────────────
        # input: fpn_dim * 3  (P3 + P4 + P5 concatenated after GAP)
        self.classifier = nn.Sequential(
            nn.Linear(fpn_dim * 3, head_dim),
            nn.LayerNorm(head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_dim, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias,   0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        c3 = self.stage1(x)
        c4 = self.stage2(c3)
        c5 = self.stage3(c4)

        p3, p4, p5 = self.fpn(c3, c4, c5)

        # Global-average-pool each scale → (B, fpn_dim)
        p3 = self.pool(p3).flatten(1)
        p4 = self.pool(p4).flatten(1)
        p5 = self.pool(p5).flatten(1)

        # Concat all scales → (B, fpn_dim * 3)
        feat = torch.cat([p3, p4, p5], dim=1)
        return self.classifier(feat)


def build_multiscale_backbone(
    backbone_name: str,
    num_classes  : int   = 2,
    fpn_dim      : int   = 256,
    head_dim     : int   = 512,
    dropout      : float = 0.4,
    pretrained   : bool  = True,
) -> MultiScaleBackbone:
    return MultiScaleBackbone(
        backbone_name = backbone_name,
        num_classes   = num_classes,
        fpn_dim       = fpn_dim,
        head_dim      = head_dim,
        dropout       = dropout,
        pretrained    = pretrained,
    )


# ─────────────────────────────────────────────────────────────────────────────
# BATCH HELPER — supports both dict and tuple loaders
# ─────────────────────────────────────────────────────────────────────────────

def unpack_batch(batch, device):
    if isinstance(batch, dict):
        images = batch["qry_im"].to(device, non_blocking=True)
        labels = batch["qry_gt"].to(device, non_blocking=True).long()
    else:
        images, labels = batch
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).long()
    return images, labels


# ─────────────────────────────────────────────────────────────────────────────
# TRAIN EPOCH
# ─────────────────────────────────────────────────────────────────────────────

def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss            = 0.0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc="Training", leave=False):
        images, labels = unpack_batch(batch, device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            logits = model(images)
            loss   = criterion(logits, labels)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item()
        all_preds.append(logits.argmax(1).detach().cpu())
        all_labels.append(labels.detach().cpu())

    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="binary", pos_label=1, zero_division=0)
    return total_loss / len(loader), acc, f1, all_preds, all_labels


# ─────────────────────────────────────────────────────────────────────────────
# EVALUATE
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss            = 0.0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        images, labels = unpack_batch(batch, device)

        with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
            logits = model(images)
            loss   = criterion(logits, labels)

        total_loss += loss.item()
        all_preds.append(logits.argmax(1).cpu())
        all_labels.append(labels.cpu())

    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    acc      = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro",  zero_division=0)
    f1_pos   = f1_score(all_labels, all_preds, average="binary", pos_label=1, zero_division=0)
    return total_loss / len(loader), acc, f1_macro, f1_pos, all_preds, all_labels


# ─────────────────────────────────────────────────────────────────────────────
# TEST (run inference + save report to disk)
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def test_model(
    model       ,
    loader      ,
    criterion   ,
    device      ,
    save_dir    : str,
    folder_name : str,
    dataset_name: str,
    class_names : tuple = ("Benign", "Malignant"),
) -> dict:
    """
    Runs inference, prints results to console, saves report to:
        save_dir / folder_name / {dataset_name}_report.txt
    """
    test_loss, test_acc, test_f1_macro, test_f1_pos, preds, labels = evaluate(
        model, loader, criterion, device
    )

    cm             = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # ── Console ───────────────────────────────────────────────────────────────
    print(f"\n{'=' * 70}")
    print(f"  {dataset_name}")
    print(f"  Loss={test_loss:.4f}  Acc={test_acc:.4f}  F1={test_f1_pos:.4f}  "
          f"Sens={sens:.4f}  Spec={spec:.4f}")
    print(f"{'=' * 70}")
    print(classification_report(labels, preds,
          target_names=list(class_names), zero_division=0))
    print("Confusion Matrix:")
    print(cm)

    # ── Disk ──────────────────────────────────────────────────────────────────
    save_report(
        save_dir      = save_dir,
        folder_name   = folder_name,
        dataset_name  = dataset_name,
        labels        = labels,
        preds         = preds,
        class_names   = class_names,
        extra_metrics = {
            "Loss"      : f"{test_loss:.4f}",
            "F1 (macro)": f"{test_f1_macro:.4f}",
        },
    )

    return {
        "loss": test_loss,
        "acc" : test_acc,
        "f1"  : test_f1_pos,
        "sens": sens,
        "spec": spec,
        "cm"  : cm,
    }


# ─────────────────────────────────────────────────────────────────────────────
# MAIN TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────

def train_model(
    model       ,
    train_loader,
    val_loader  ,
    device      ,
    save_dir    : str,
    epochs      : int   = 50,
    patience    : int   = 10,
    lr          : float = 1e-4,
    weight_decay: float = 1e-4,
    class_names : tuple = ("Benign", "Malignant"),
):
    """
    Trains model, saves best checkpoint, writes training_log.txt,
    and saves val report each time val F1 improves.

    Returns: model (loaded from best ckpt), best_val_f1, history, criterion
    """
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, "best_model.pth")
    log_path  = os.path.join(save_dir, "training_log.txt")

    criterion = AsymmetricFocalLoss(
        gamma_neg=2.0, gamma_pos=1.0, alpha=0.8, label_smoothing=0.0,
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    scaler    = GradScaler(enabled=(device.type == "cuda"))

    best_val_f1      = 0.0
    patience_counter = 0
    history          = {"train_f1": [], "val_f1": [], "train_loss": [], "val_loss": []}

    with open(log_path, "w") as lf:
        lf.write(f"Training log — {save_dir}\n")
        lf.write(f"lr={lr}  wd={weight_decay}  epochs={epochs}  patience={patience}\n\n")

    for epoch in range(epochs):
        t0 = time.time()
        print(f"\n{'=' * 70}")
        print(f"  EPOCH {epoch + 1}/{epochs}")
        print(f"{'=' * 70}")

        # ── Train ─────────────────────────────────────────────────────────────
        train_loss, train_acc, train_f1, tr_preds, tr_labels = train_epoch(
            model, train_loader, optimizer, criterion, device, scaler
        )

        # ── Validate ──────────────────────────────────────────────────────────
        val_loss, val_acc, val_f1_macro, val_f1_pos, val_preds, val_labels = evaluate(
            model, val_loader, criterion, device
        )

        scheduler.step()

        history["train_f1"].append(train_f1)
        history["val_f1"].append(val_f1_pos)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print_epoch_summary("train", train_loss, train_acc, train_f1,
                            tr_preds, tr_labels, class_names=class_names)
        print_epoch_summary("val",   val_loss,   val_acc,   val_f1_pos,
                            val_preds, val_labels, class_names=class_names)

        ep_time = round(time.time() - t0, 2)
        lr_now  = optimizer.param_groups[0]["lr"]
        print(f"\n  LR={lr_now:.2e}  Time={ep_time}s")

        # Append to log
        with open(log_path, "a") as lf:
            lf.write(
                f"Epoch {epoch+1:03d}  "
                f"train_loss={train_loss:.4f}  train_f1={train_f1:.4f}  "
                f"val_loss={val_loss:.4f}  val_f1={val_f1_pos:.4f}  "
                f"lr={lr_now:.2e}  time={ep_time}s\n"
            )

        # ── Checkpoint + val report ────────────────────────────────────────────
        if val_f1_pos > best_val_f1:
            best_val_f1 = val_f1_pos
            torch.save(model.state_dict(), save_path)
            print(f"  [best] Saved  val_f1={best_val_f1:.4f}")
            patience_counter = 0

            # Save val report at this epoch
            save_report(
                save_dir      = save_dir,
                folder_name   = "vindr_val",
                dataset_name  = f"VinDr-Val-Epoch{epoch+1:03d}",
                labels        = val_labels,
                preds         = val_preds,
                class_names   = class_names,
                extra_metrics = {
                    "Epoch"    : str(epoch + 1),
                    "Val Loss" : f"{val_loss:.4f}",
                    "Val F1"   : f"{val_f1_pos:.4f}",
                },
            )
        else:
            patience_counter += 1
            print(f"  No improvement  patience={patience_counter}/{patience}  "
                  f"best={best_val_f1:.4f}")
            if patience_counter >= patience:
                print("  Early stopping.")
                break

    # Load best checkpoint
    model.load_state_dict(torch.load(save_path, map_location=device))
    print(f"\n  Loaded best checkpoint  val_f1={best_val_f1:.4f}")
    return model, best_val_f1, history, criterion


# ─────────────────────────────────────────────────────────────────────────────
# EXPERIMENT RUNNER
# ─────────────────────────────────────────────────────────────────────────────

def run_experiments(
    train_loader   ,
    val_loader     ,
    test_loader    ,
    inbreast_loader,
    epochs        : int   = 60,
    patience      : int   = 15,
    lr            : float = 1e-4,
    weight_decay  : float = 1e-4,
    base_dir      : str   = "results_multiscale",
):
    """
    Runs both backbones sequentially.

    Disk layout:
        base_dir/
            all_results_summary.txt
            efficientnet_b3/
                best_model.pth
                training_log.txt
                summary.txt
                vindr_val/        VinDr-Val-Epoch{N}_report.txt
                vindr_test/       VinDr-Test_report.txt
                inbreast_test/    INbreast_report.txt
            convnext_base/
                (same)
    """
    set_seed(2024)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    backbones = [
        # "efficientnet_b3", 
                 "convnext_base"]
    results: Dict[str, Dict] = {}
    os.makedirs(base_dir, exist_ok=True)

    for backbone_name in backbones:
        print(f"\n{'#' * 80}")
        print(f"#  {backbone_name}")
        print(f"{'#' * 80}")

        save_dir = os.path.join(base_dir, backbone_name)
        os.makedirs(save_dir, exist_ok=True)

        model = build_multiscale_backbone(
            backbone_name = backbone_name,
            num_classes   = 2,
            fpn_dim       = 256,
            head_dim      = 512,
            dropout       = 0.4,
            pretrained    = True,
        ).to(device)

        total_p     = sum(p.numel() for p in model.parameters())
        trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Params total={total_p:,}  trainable={trainable_p:,}")

        # ── Train ─────────────────────────────────────────────────────────────
        model, best_val_f1, history, criterion = train_model(
            model        = model,
            train_loader = train_loader,
            val_loader   = val_loader,
            device       = device,
            save_dir     = save_dir,
            epochs       = epochs,
            patience     = patience,
            lr           = lr,
            weight_decay = weight_decay,
            class_names  = ("Benign", "Malignant"),
        )
        print(f"\n  Best Val F1: {best_val_f1:.4f}")

        # ── VinDr Test ────────────────────────────────────────────────────────
        # → save_dir/vindr_test/VinDr-Test_report.txt
        vindr_res = test_model(
            model        = model,
            loader       = test_loader,
            criterion    = criterion,
            device       = device,
            save_dir     = save_dir,
            folder_name  = "vindr_test",
            dataset_name = "VinDr-Test",
            class_names  = ("Benign", "Malignant"),
        )

        # ── INbreast Test ─────────────────────────────────────────────────────
        # → save_dir/inbreast_test/INbreast_report.txt
        inbreast_res = test_model(
            model        = model,
            loader       = inbreast_loader,
            criterion    = criterion,
            device       = device,
            save_dir     = save_dir,
            folder_name  = "inbreast_test",
            dataset_name = "INbreast",
            class_names  = ("Benign", "Malignant"),
        )

        results[backbone_name] = {
            "best_val_f1": best_val_f1,
            "vindr"      : vindr_res,
            "inbreast"   : inbreast_res,
            "history"    : history,
        }

        # ── Per-backbone summary file ─────────────────────────────────────────
        summary_path = os.path.join(save_dir, "summary.txt")
        with open(summary_path, "w") as sf:
            sf.write(f"Model : {backbone_name}\n")
            sf.write(f"Best Val F1 : {best_val_f1:.4f}\n\n")
            sf.write("VinDr-Test\n")
            sf.write(f"  Acc={vindr_res['acc']:.4f}  F1={vindr_res['f1']:.4f}  "
                     f"Sens={vindr_res['sens']:.4f}  Spec={vindr_res['spec']:.4f}\n\n")
            sf.write("INbreast-Test\n")
            sf.write(f"  Acc={inbreast_res['acc']:.4f}  F1={inbreast_res['f1']:.4f}  "
                     f"Sens={inbreast_res['sens']:.4f}  Spec={inbreast_res['spec']:.4f}\n\n")
            sf.write("Val F1 per epoch:\n")
            for i, v in enumerate(history["val_f1"]):
                sf.write(f"  Epoch {i+1:03d}: {v:.4f}\n")
        print(f"  [saved] {summary_path}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # ── Console summary ───────────────────────────────────────────────────────
    print(f"\n{'=' * 90}")
    print("FINAL RESULTS SUMMARY")
    print(f"{'=' * 90}")
    print(f"{'Model':<20}  {'ValF1':>7}  {'VinDr-F1':>9}  "
          f"{'Sens':>6}  {'Spec':>6}  "
          f"{'INb-F1':>7}  {'Sens':>6}  {'Spec':>6}")
    print("-" * 90)
    for name, r in results.items():
        print(
            f"{name:<20}  "
            f"{r['best_val_f1']:>7.4f}  "
            f"{r['vindr']['f1']:>9.4f}  "
            f"{r['vindr']['sens']:>6.4f}  "
            f"{r['vindr']['spec']:>6.4f}  "
            f"{r['inbreast']['f1']:>7.4f}  "
            f"{r['inbreast']['sens']:>6.4f}  "
            f"{r['inbreast']['spec']:>6.4f}"
        )
    print("=" * 90)

    # ── Overall summary file ──────────────────────────────────────────────────
    overall_path = os.path.join(base_dir, "all_results_summary.txt")
    with open(overall_path, "w") as of:
        of.write("ALL MODELS — FINAL RESULTS\n")
        of.write("=" * 70 + "\n\n")
        for name, r in results.items():
            of.write(f"Model: {name}\n")
            of.write(f"  Best Val F1  : {r['best_val_f1']:.4f}\n")
            of.write(f"  VinDr-Test   : F1={r['vindr']['f1']:.4f}  "
                     f"Sens={r['vindr']['sens']:.4f}  "
                     f"Spec={r['vindr']['spec']:.4f}\n")
            of.write(f"  INbreast-Test: F1={r['inbreast']['f1']:.4f}  "
                     f"Sens={r['inbreast']['sens']:.4f}  "
                     f"Spec={r['inbreast']['spec']:.4f}\n\n")
    print(f"\n[saved] {overall_path}")

    return results


# ─────────────────────────────────────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────────────────────────────────────

results = run_experiments(
    train_loader    = train_loader,
    val_loader      = val_loader,
    test_loader     = test_loader,
    inbreast_loader = inbreast_loader,
    epochs          = 60,
    patience        = 15,
    lr              = 1e-4,
    weight_decay    = 1e-4,
    base_dir        = "Thesis_updated_results/multiscale_binary_512",
)

In [ ]:
import torch
import torchvision.models as tvm

x = torch.zeros(1, 3, 512, 512)

# ── EfficientNet-B3 ───────────────────────────────────────────────────────────
net = tvm.efficientnet_b3(weights=None)
print("EfficientNet-B3  (input 512x512)")
for i in range(len(net.features)):
    out = net.features[:i+1](x)
    print(f"  features[:{i+1}]  ->  ch={out.shape[1]}  spatial={out.shape[2]}x{out.shape[3]}")

print()

# ── ConvNeXt-Base ─────────────────────────────────────────────────────────────
net2 = tvm.convnext_base(weights=None)
print("ConvNeXt-Base  (input 512x512)")
for i in range(len(net2.features)):
    out2 = net2.features[:i+1](x)
    print(f"  features[:{i+1}]  ->  ch={out2.shape[1]}  spatial={out2.shape[2]}x{out2.shape[3]}")